In [ ]:
import ROOT
import emm

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Get data
data_tree = emm.get_data(sort_and_index=True, tree=True)

x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 4000)
index=ROOT.RooRealVar("index", "index", 0, 0, 1e6)

data = ROOT.RooDataSet("data", "data", ROOT.RooArgSet(x, index), ROOT.RooFit.Import(data_tree))

# Initial Conditions

In [ ]:
pset = {
    'raw_rate_0': (0.3, 2.5, 50),
    'raw_rate_1': (0.3, 2.5, 50),
}

model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))

df = emm.scan_parameters(
    model, data, pset,
    constant=False, # Not fixing parameters
    # use_condor=False, n_batches=8, cache_name="2D_grid_restarts", remake_cache=True,
    use_condor=True, n_batches=64, cache_name="2D_grid_restarts",# remake_cache=True,
)

fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False,)
print(f"Minimum NLL: {df['nll'].min()}")
print(f"Best fit parameters: {df.loc[df['nll'].idxmin()]}")

In [ ]:
# Plot the distribution of the NLL
fig, ax = plt.subplots()
m = df['nll'].min()
n, bins, patches = ax.hist(df['nll'], range=(m, m+0.001), bins=50, density=True)
ax.set_xlabel("NLL")
ax.set_ylabel("Density")

In [ ]:
pset = {
    'raw_rate_0': (0.3, 3, 40),
    'raw_rate_1': (0.3, 3, 40),
    'raw_rate_2': (0.3, 3, 40),
}

model = emm.ExponentialMixtureModel(x, 3, data_mean=data.mean(x))

df = emm.scan_parameters(
    model, data, pset,
    constant=False, # Not fixing parameters
    use_condor=True, n_batches=64, cache_name="3D_grid_restarts",# remake_cache=True,
)

fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False, worst_case=False)
print(f"Minimum NLL: {df['nll'].min()}")
print(f"Best fit parameters: {df.loc[df['nll'].idxmin()]}")

In [ ]:
# Plot the distribution of the NLL
fig, ax = plt.subplots()
m = df['nll'].min()
n, bins, patches = ax.hist(
    df['nll'],
    # range=(m, m+0.005),
    # range=(31090, 31100),
    bins=50,
    # density=True
)
ax.set_xlabel("NLL")
# ax.set_ylabel("Density")

# Does it matter if we get stuck in a local minimum?

In [ ]:
# Use toys to estimate the NLL uncertainty due to data statistics
best_initial = emm.get_initial(2)
worst_initial = emm.get_initial(2, worst_case=True)

model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))
defaults = {param.GetName(): param.getVal() for param in model.params}

nll = model.pdf.createNLL(data)

# Get worst case nll
model.set_params(worst_initial)
model.pdf.fitTo(data)
nll_worst = model.pdf.createNLL(data)

# Gest best case nll
model.set_params(best_initial)
model.pdf.fitTo(data)
nll_best = model.pdf.createNLL(data)

# Generate toys and calculate the different in NLL between the best and worst initializations
n_toys = 100
nlls_global = []
nlls_local = []
for i in range(n_toys):
    toy = model.pdf.generate(ROOT.RooArgSet(x), data.numEntries())
    # nll = model.pdf.createNLL(toy)
    i_model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x))
    i_model.set_params(initials)
    nll = i_model.pdf.createNLL(toy)
    fit_result = model.pdf.fitTo(toy)
    nlls.append(nll.getVal())

In [ ]:
# Plot the distribution of the nlls and print some statistics
plt.hist(nlls, bins=10)
plt.xlabel("NLL")
plt.ylabel("Number of toys")
plt.title("NLL distribution from toys")
plt.show()
print(f"Mean NLL: {np.mean(nlls)}")
print(f"Std NLL: {np.std(nlls)}")

In [ ]:
initials = {
    'raw_rate_0': 1.295,
    'raw_rate_1': 0.584,
    'raw_rate_2': 1.579,
}

In [ ]:
model = emm.ExponentialMixtureModel(x, 3, data_mean=data.mean(x))
model.set_params(initials)
emm.fit_and_plot(model, data, x)

In [ ]:
# Add two events in the tail
data_copy = data.Clone()
# vals_to_add = [2500]*50
vals_to_add = np.random.normal(3800, 30, size=1)
for val in vals_to_add:
    x.setVal(val)
    index.setVal(data_copy.numEntries())
    data_copy.add(ROOT.RooArgSet(x, index))

print(f"Data entries (with tail events): {data_copy.numEntries()}")

In [ ]:
# model = emm.ExponentialMixtureModel(x, 3, data_mean=data_copy.mean(x))
models = [emm.ExponentialMixtureModel(x, k, data_mean=data_copy.mean(x)) for k in range(2, 5)]
model_labels = [f"{k} Mixtures" for k in range(2, 5)]
fit_results = [m.pdf.fitTo(data_copy, ROOT.RooFit.Save()) for m in models]
emm.plot_fits(data_copy, x, models, model_labels, fit_results,)
# model.set_params(initials)
# model.set_param('raw_rate_2', 0, constant=True)
# emm.fit_and_plot(model, data_copy, x)

In [ ]:
pset = {
    'raw_rate_0': (0.3, 3, 20),
    'raw_rate_1': (0.3, 3, 20),
}

def model_primitive(x, data):
    model = emm.ExponentialMixtureModel(x, 4, data_mean=data.mean(x), ordered_rates=True)
    return model

df = emm.profile_model(x, model_primitive, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset, plot_contours=False,)

In [ ]:
model_2 = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x), use_normalization_construction=True)
emm.fit_and_plot(model_2, data, x, minos=True)

In [ ]:
pset = {
    # 'raw_rate_0': (0.01, 5, 20),
    # 'raw_rate_1': (0.3, 2.5, 20),
    'raw_rate_2': (0.3, 5, 20),
    'raw_rate_3': (0.3, 5, 20)
}

model_2_params = {}
for i in range(model_2.n_components):
    r = model_2.raw_rates[i]
    r_nominal = r.getVal()
    r_up = r.getErrorHi()
    r_down = r.getErrorLo()
    model_2_params[f'raw_rate_{i}'] = (r_nominal, r_nominal-r_down, r_nominal+r_up)

def model_primitive(x, data):
    model = emm.ExponentialMixtureModel(x, 4, data_mean=data.mean(x), use_normalization_construction=True, **model_2_params)
    return model

df = emm.profile_model(x, model_primitive, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset)

In [ ]:
#print the minimum
print("Minimum value of the model:", df['nll'].min())
print(f"Parameters at minimum: \n{df.loc[df['nll'].idxmin()]}")
    

In [ ]:
#print the minimum
print("Minimum value of the model:", df['nll'].min())
print(f"Parameters at minimum: \n{df.loc[df['nll'].idxmin()]}")

In [ ]:

grid = np.linspace(0, 10, 100)

def fit_model(supports):
    model = emm.ExponentialMixtureModel(x, len(supports), data_mean=data.mean(x), use_normalization_construction=True)
    for i, support in enumerate(supports):
        model.raw_rates[i].setVal(support)
        model.raw_rates[i].setConstant(True)
    
    model.pdf.fitTo(data)
    weights = [p.getVal() for p in model.weights]

    nll = model.pdf.createNLL(data)
    return {
        "nll": nll.getVal(),
        "weights": weights,
        "supports": supports
    }

# Randomly search through the parameter space, if a new nll is found 

In [ ]:
""